# ALD_평가 — 내부 합성 Test 성능 / 원본 Test 추론속도
기존 DEP 전이 가중치를 로드해 합성 Test 성능과 원본 입력의 전체 요청 시간을 각각 확인합니다. 학습은 명시적으로 켤 때만 수행합니다. 합성 정확도와 원본 속도는 서로 다른 입력 조건입니다.


In [ ]:
from pathlib import Path
import sys, importlib, os
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir())
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
import pecvd.experiments.pipeline_settings as pipeline_settings
pipeline_settings=importlib.reload(pipeline_settings)
PIPELINE_SETTINGS=pipeline_settings.configure_pipeline(ROOT)
if PIPELINE_SETTINGS.get('decision_system')!='fusion_answers':
    raise RuntimeError('These experiments require the trained fusion_answers model.')
from pecvd.experiments.questions import PRIMARY_GOAL_VERSION
from pecvd.experiments.fusion_answers import SYSTEM as GEVIS_SYSTEM
print({'model':GEVIS_SYSTEM,'training_profile':PIPELINE_SETTINGS['training_profile'],'weights':'existing STEP1/2/3'})


In [ ]:
import json
import pandas as pd
from pecvd.experiments import fast_experiments as fast, peald_training as peald
from pecvd.experiments import peald_encoder_routing as routing
from pecvd.process_config import load_process_config
RUNTIME=fast.load_runtime(ROOT)
PEALD_CONFIG_PATH=ROOT/'.cache/external/peald/process.json'
if not PEALD_CONFIG_PATH.is_file():
    raise FileNotFoundError('Prepare and review the explicit local PEALD mapping before EXP6.')
PEALD_CONFIG=load_process_config(PEALD_CONFIG_PATH)
CONFIG=peald.PealdConfig(**json.loads((ROOT/'config/experiments/peald_fast.json').read_text(encoding='utf-8')))
ENCODER_SELECTION_MODE='single'  # Previous DEP path; 'validation' is a deferred comparison experiment.
SOURCE_ENCODER_STEP='DEP.'  # Shared frozen source encoder in the default single mode.
SOURCE_ENCODER_STEPS=None  # None considers every loaded PECVD Step encoder.
TARGET_STEPS=None
RUN_PEALD_TRAINING=False  # Reuse exact checkpoints first; False refuses fitting if missing.
CENTER_BATCH_SIZE=8
BENCHMARK_REPEATS=3  # Smoke benchmark; increase for a fixed acceptance protocol.
BENCHMARK_WARMUP=1
if ENCODER_SELECTION_MODE not in ('single','validation'):
    raise ValueError('Choose single or validation encoder selection')
SOURCE_ENCODER=RUNTIME.encoders[SOURCE_ENCODER_STEP][0]
PEALD_API=routing if ENCODER_SELECTION_MODE=='validation' else peald
PREPARATION_PATCH_LENGTH=(routing.preparation_patch_length(RUNTIME,SOURCE_ENCODER_STEPS)
    if ENCODER_SELECTION_MODE=='validation' else SOURCE_ENCODER.patch_len)
print({'encoder_selection_mode':ENCODER_SELECTION_MODE,'source_encoder_step':SOURCE_ENCODER_STEP,
       'preparation_patch_length':PREPARATION_PATCH_LENGTH,'source_weights_updated':False})
if ENCODER_SELECTION_MODE=='single':
    print({'common_head_epochs':CONFIG.epochs,'common_head_fit_seconds':CONFIG.max_seconds,
           'step_heads_enabled':CONFIG.step_heads_enabled,'step_head_epochs':CONFIG.step_epochs,
           'checkpoint_policy':'reuse exact existing identity; fit only if missing and allowed'})
else:
    print({'experiment_status':'deferred comparison; not an accepted improvement over the previous DEP model',
           'candidates':routing.source_steps(RUNTIME,SOURCE_ENCODER_STEPS),
           'candidate_epochs':CONFIG.step_epochs,'candidate_fit_seconds':CONFIG.step_max_seconds,
           'candidate_budget_scope':'per compatible source encoder and target Step; feature preparation extra'})

SHOW_DETAILS=False
RUN_ORIGINAL_REPLAY=False
RUN_BOOTSTRAP=False


## 1. 전처리와 제외·검토 사유 확인

원본 CSV는 수정하지 않습니다. exact duplicate만 제거하고 값이 충돌하는 동일 시각 행은 격리합니다.
시간 역행·회차 ID 재등장·회차 내 recipe 변경은 회차의 해석이 불명확하므로 해당 회차를 격리합니다.
샘플 간격은 TRAIN의 중앙값·MAD로 허용 범위를 계산합니다. 범위를 벗어난 경계에서 구간을 끊습니다.

정규화는 TRAIN 중앙값·IQR을 사용하고, Step 내 IQR이 0이면 TRAIN 전체의 센서 표준편차로 보완합니다.
TRAIN에서 스케일을 알 수 없는 센서는 제외 사유를 표시합니다. Test로 스케일을 다시 맞추지 않습니다.
신호가 크게 벗어난 구간은 **삭제하지 않는 검토 후보**입니다. 정상/고장 정답으로 학습하지 않습니다.
지원되지 않는 phase, 짧은 조각, window 뒤 남는 행도 표에서 확인합니다.


In [ ]:
PREPARED=peald.prepare(ROOT,PEALD_CONFIG,patch_length=PREPARATION_PATCH_LENGTH,config=CONFIG)
if SHOW_DETAILS:
    for _name,_table in PREPARED.tables.items():
        print(_name);display(_table)
if ENCODER_SELECTION_MODE=='validation':
    PEALD_RUNTIME=routing.runtime_for(RUNTIME,PREPARED,candidates=SOURCE_ENCODER_STEPS,baseline=SOURCE_ENCODER_STEP)
else:
    PEALD_RUNTIME=peald.runtime_for(RUNTIME,PREPARED,source_step=SOURCE_ENCODER_STEP)
print({'run_split_counts':{s:len(r) for s,r in PREPARED.split_runs.items()},
       'supported_steps':list(PREPARED.windows),'target_steps_source_trained':False})


## 2. 기존 DEP 경로의 PEALD 공통·Step 헤드

기본 `single`은 모든 PEALD Step에 PECVD `DEP.` 인코더를 사용합니다. DEP patch 길이로 입력을 준비하고, 인코더·Fusion을 동결한 채 전체 Step의 TRAIN으로 공통 헤드를 학습합니다. 이후 공통 헤드에서 시작한 Step별 후보를 Validation으로 채택합니다. Test는 선택에 사용하지 않습니다. 설정상 공통 헤드는 최대 40 epoch·300초, Step 후보는 최대 16 epoch·30초입니다.

정확한 identity의 체크포인트가 있으면 재사용합니다. 없으면 `RUN_PEALD_TRAINING=True`일 때만 학습합니다. 기본 False에서는 기존 가중치만 사용합니다. 입력·설정이 달라진 가중치를 강제로 로딩하지 않습니다. 기존 가중치와 결과는 덮지 않습니다.

### 추후 비교 실험: validation

`ENCODER_SELECTION_MODE='validation'`으로 바꾸면 호환 PECVD 후보별 새 Step 헤드를 TRAIN에서 학습하고 Validation으로 인코더·헤드 조합을 선택합니다. 같은 window를 native patch로 처리하지 못하는 후보는 이유를 표시합니다. Test로 선택하지 않습니다.

이 경로는 이전 공통 DEP 모델과 학습 구조·예산·window grid가 다를 수 있고, 이전 학습 완료 모델을 그대로 대조군에 포함하지 않습니다. 현재 개선안으로 채택하지 않으며, 공정한 비교에는 기존 모델을 보존한 동일 입력·학습 예산의 별도 실험이 필요합니다.


In [ ]:
TRAINING=PEALD_API.train_or_load(ROOT,PEALD_RUNTIME,steps=TARGET_STEPS,config=CONFIG,
    allow_training=RUN_PEALD_TRAINING)
if SHOW_DETAILS:
    display(pd.DataFrame(TRAINING['history']))
    display(pd.json_normalize(TRAINING['step_head_selection']))
if ENCODER_SELECTION_MODE=='validation':
    display(pd.DataFrame(TRAINING['candidates']))
    print({'selection_manifest':TRAINING['selection_manifest'],'selection_split':'validation',
           'test_used_for_selection':False,'source_weights_updated':False})
else:
    print({'checkpoint':str(PEALD_RUNTIME.module.FUSION_ANSWER_PATH),
           'trained_steps':TRAINING['trained_steps'],'source_weights_updated':False})


## 3. 기존 paired synthetic 진단 보존

정확한 주입 전 원본을 함께 주는 비교용 진단입니다. 공통 TRAIN 참조 적용 시험의 정확도를 대신하지 않습니다.
PEALD Test 원본 파형에 통제 변화를 추가해 모델이 맞혔는지 채점합니다.
BOOLEAN precision·recall·F1이 변화 탐지 지표입니다. CHOICE·ORDINAL은 유형·크기 해석 지표입니다.
정답 라벨은 주입 기록에서 나오며 모델 입력에는 포함하지 않습니다.
기존에 확인한 Test와 이번 결과는 개발용 비교이며, 실제 공정 고장 성능을 뜻하지 않습니다.

세 답변 모두 Fast의 평가 대상입니다. BOOLEAN 양성 F1과 CHOICE·ORDINAL macro F1을 각각 목표 0.96과 참고 비교하며, 답변별 최종 수락 정의는 확정해야 합니다.
평균 50 ms 목표는 이 정확도 표로 판정할 수 없습니다. 대상 Edge의 수집 경계부터 결과 사용까지 별도로 측정해야 합니다.
`step_head_selection`은 Validation의 채택 결과입니다. Test 결과로 헤드를 고르지 않습니다. Step별 오탐률과 동일/타 센서 변화 대조군의 오탐률은 SHOW_DETAILS=True로 확인합니다.
오탐률은 알려진 음성 사례에서 FP/(FP+TN)이며, 음성이 없는 Step은 평가 불가입니다. 원본 공정의 실제 오탐률과 구분합니다.


In [ ]:
from pecvd.experiments import fast_assessment
PAIRED_BASELINE=PEALD_API.evaluate(PEALD_RUNTIME,steps=TARGET_STEPS,config=CONFIG)
REPORT=PAIRED_BASELINE;SUMMARY=REPORT.summary;DETAILS=REPORT.details
print('Labelled paired synthetic Test; not original-run fault accuracy.')
PAIRED_ASSESSMENT=fast_assessment.compact(REPORT,display,evidence_scope='PEALD_paired_synthetic_Test',show_details=SHOW_DETAILS)
if RUN_BOOTSTRAP:
    from pecvd.experiments import evaluation_audit
    display(evaluation_audit.run_bootstrap(REPORT.details))


## 4. 기존 Test의 공통 TRAIN 참조 추론

새 입력과 동일하게 고정된 TRAIN 참조를 선택하고 모델의 모든 선택 센서 답변을 반환합니다. 독립 정답이 없는 원본 회차 비교의 정확도를 만들지 않습니다. 앞의 paired synthetic 표는 기존 정확도 비교용 진단이며 이 운영 조건의 정확도를 대신하지 않습니다.


In [ ]:
from pecvd.experiments import online_inference as online, inference_benchmark as inference_timing
INFERENCE=online.InferenceSession(PEALD_RUNTIME,center_batch_size=CENTER_BATCH_SIZE)
ONLINE_CASES=inference_timing.peald_cases(PEALD_RUNTIME,steps=TARGET_STEPS,config=CONFIG)
if RUN_ORIGINAL_REPLAY:
    ORIGINAL=online.run_cases(INFERENCE,ONLINE_CASES,domain='PEALD',progress=print)
    fast_assessment.compact(ORIGINAL,display,evidence_scope='PEALD_original_replay',predictions=18,show_details=SHOW_DETAILS)


## 5. 공통 요청 경로의 ALD 속도

외부 CSV 예측과 같은 InferenceSession.process_window를 사용합니다. 한 요청은 완성된 Step 모델 입력 구간 하나에 대해 모든 선택 센서의 세 답변을 반환합니다. 모델·TRAIN 참조 색인은 시작할 때 준비하고, 요청 안에서 전처리·참조 선택·인코더·Fusion·헤드·결과 구성을 측정합니다. 같은 구간은 한 번만 인코딩합니다.

평균·p95·p99와 50 ms 초과 비율을 확인합니다. 파일 읽기·구간 완성 전 대기·구간 분리·전송·대기열은 제외하므로 물리 cycle 전체 또는 Edge KPI 달성을 인증하지 않습니다. 이전 한 센서 3답변 시간과 처리 단위가 달라 직접 개선율로 비교하지 않습니다.


In [ ]:
BENCHMARK=inference_timing.benchmark(PEALD_RUNTIME,ONLINE_CASES,
    repeats=BENCHMARK_REPEATS,warmup=BENCHMARK_WARMUP,domain='PEALD',session=INFERENCE,progress=print)
LATENCY_DETAILS=BENCHMARK.details
BENCHMARK_ASSESSMENT=fast_assessment.compact(BENCHMARK,display,evidence_scope='PEALD_original_repeated_requests',show_details=SHOW_DETAILS)
